In [1]:
##Contact Book Application Project

import csv

# 1.Contact Class =====
class Contact:    
    def __init__(self, name, phone, email, address):
        self.name = str(name).strip()
        self.phone = str(phone).strip()
        self.email = str(email).strip()
        self.address = str(address).strip()
    
    def update(self, phone=None, email=None, address=None):
        try:
            if phone:
                self.phone = str(phone).strip()
            if email:
                self.email = str(email).strip()
            if address:
                self.address = str(address).strip()
            return True
        except:
            return False
    
    def __str__(self):
        return f"Name: {self.name}\nPhone: {self.phone}\nEmail: {self.email}\nAddress: {self.address}"


# 2. Data Storage Module =====
class DataStorage:
    def __init__(self):
        self.contacts_list = []
        self.phone_dict = {}
    
    def insert(self, contact):
        try:
            if contact.phone in self.phone_dict:
                return False, "This Phone number already exists"
            if not self.validate_phone(contact.phone):
                return False, "Phone must be 7 to 15 digits"
            if '@' not in contact.email:
                return False, "Email must contain '@'"
            pos = 0
            while pos < len(self.contacts_list) and self.contacts_list[pos].name.lower() < contact.name.lower():
                pos += 1
            
            self.contacts_list.insert(pos, contact)
            self.phone_dict[contact.phone] = contact
            return True, "Contact added successfully"
        except Exception as e:
            return False, f"Error: {str(e)}"
    
    def delete(self, phone):
        try:
            if phone not in self.phone_dict:
                return False, "Phone not found"
            contact = self.phone_dict[phone]
            for i, c in enumerate(self.contacts_list):
                if c.phone == phone:
                    del self.contacts_list[i]
                    break
            del self.phone_dict[phone]
            return True, "Contact deleted successfully"
        except Exception as e:
            return False, f"Error: {str(e)}"
    
    def search(self, search_term):
        if search_term in self.phone_dict:
            return self.phone_dict[search_term], "Found by phone"
        
        results = []
        search_lower = search_term.lower()
        for contact in self.contacts_list:
            if search_lower in contact.name.lower():
                results.append(contact)
        
        if results:
            return results, f"Found {len(results)} contacts by name"
        return None, "No contact found"
    
    def get_sorted(self):
        return self.contacts_list
    
    def get_count(self):
        return len(self.contacts_list)
    
    def validate_phone(self, phone):
        phone = phone.replace(" ", "")
        return phone.isdigit() and 7 <= len(phone) <= 15


# 3 File Handling Module =====
class FileHandler:
    def __init__(self, filename="contacts.csv"):
        self.filename = filename
    
    def read_contacts(self):
        contacts = []
        try:
            with open(self.filename, 'r') as file:
                reader = csv.reader(file)
                next(reader, None) 
                for row in reader:
                    if len(row) >= 5:
                        contact = Contact(row[1], row[2], row[3], row[4])
                        contacts.append(contact)
            return True, 
        except FileNotFoundError:
            return True, 
        except:
            return False, "Error reading file"
    
    def write_contacts(self, contacts):
        try:
            with open(self.filename, 'w', newline='') as file:
                writer = csv.writer(file)
                writer.writerow(['No', 'Name', 'Phone', 'Email', 'Address']) 
                for i, contact in enumerate(contacts, 1):
                    writer.writerow([i, contact.name, contact.phone, contact.email, contact.address])
            return True, f"Saved {len(contacts)} contacts"
        except:
            return False, "Error saving"


# 4 ContactBook Manager =====
class ContactBookManager:
    def __init__(self):
        self.storage = DataStorage()
        self.file_handler = FileHandler()
        self._load_initial()
    
    def _load_initial(self):
        self.file_handler.read_contacts()  
    
    def save_contacts(self):
        contacts = self.storage.get_sorted()
        return self.file_handler.write_contacts(contacts)
    
    def add_contact(self, name, phone, email, address):
        contact = Contact(name, phone, email, address)
        return self.storage.insert(contact)
    
    def delete_contact(self, phone):
        return self.storage.delete(phone)
    
    def update_contact(self, old_phone, **updates):
        result, msg = self.storage.search(old_phone)
        if not result:
            return False, "Contact not found"
        
        contact = result if isinstance(result, Contact) else result[0]
        new_phone = updates.get('phone')
        if new_phone and new_phone != old_phone:
            check_result, _ = self.storage.search(new_phone)
            if check_result:
                return False, "New phone number already exists"
        
        contact.update(phone=updates.get('phone'), email=updates.get('email'), address=updates.get('address'))
        
        if new_phone and new_phone != old_phone:
            del self.storage.phone_dict[old_phone]
            self.storage.phone_dict[new_phone] = contact
        
        return True, "Contact updated successfully"
    
    def search_contact(self, search_term):
        return self.storage.search(search_term)
    
    def display_contacts(self):
        contacts = self.storage.get_sorted()
        count = self.storage.get_count()
        return contacts, count


# 5 Menu/UI Module =====
def main_menu():
    manager = ContactBookManager()
    
    while True:
        print("\n" + "="*50)
        print("CONTACT BOOK APPLICATION")
        print("="*50)
        print("1. Add New Contact")
        print("2. View All Contacts (Sorted)")
        print("3. Search Contact")
        print("4. Update Contact")
        print("5. Delete Contact")
        print("6. Save & Exit")
        print("="*50)
        
        choice = input("Enter your choice (1-6): ").strip()
        
        if choice == "1":
            name = input("Name: ").strip()
            phone = input("Phone: ").strip()
            email = input("Email: ").strip()
            address = input("Address: ").strip()
            
            if not all([name, phone, email, address]):
                print("\n ERROR: All fields are required")
                continue
            
            success, message = manager.add_contact(name, phone, email, address)
            print("\n" + ("SUCCESS: " + message if success else "ERROR: " + message))
            
        elif choice == "2":
            contacts, count = manager.display_contacts()
            if count == 0:
                print("\n No contacts to display")
                continue
            print(f"Total Contacts: {count}")
            for i, contact in enumerate(contacts, 1):
                print(f"\nContact {i}:\n{contact}")
            
        elif choice == "3":
            search_term = input("Enter phone number or name: ").strip()
            if not search_term:
                print("\n ERROR: Please enter a search term")
                continue
            result, message = manager.search_contact(search_term)
            print(f"\n{message}")
            if result:
                if isinstance(result, list):
                    for contact in result:
                        print(f"\n{contact}")
                else:
                    print(f"\n{result}")
            
        elif choice == "4":
            phone = input("Enter phone number of contact to update: ").strip()
            if not phone:
                print("\n ERROR: Phone number is required")
                continue
            result, msg = manager.search_contact(phone)
            if not result:
                print(f"ERROR: {msg}")
                continue
            contact = result if isinstance(result, Contact) else result[0]
            print(f"\nCurrent details:\n{contact}")
            new_phone = input("New phone: ").strip()
            new_email = input("New email: ").strip()
            new_address = input("New address: ").strip()
            updates = {}
            if new_phone: updates['phone'] = new_phone
            if new_email: updates['email'] = new_email
            if new_address: updates['address'] = new_address
            if not updates: 
                print("No changes made")
                continue
            success, message = manager.update_contact(phone, **updates)
            print("\n" + ("SUCCESS: " + message if success else "ERROR: " + message))
            
        elif choice == "5":
            phone = input("Enter phone number to delete: ").strip()
            if not phone:
                print("\n ERROR: Phone number is required")
                continue
            result, msg = manager.search_contact(phone)
            if not result:
                print(f"ERROR: {msg}")
                continue
            contact = result if isinstance(result, Contact) else result[0]
            confirm = input(f"Are you sure you want to delete {contact.name}? (yes/no): ").strip().lower()
            if confirm in ['yes', 'y']:
                success, message = manager.delete_contact(phone)
                print("\n" + ("SUCCESS: " + message if success else "ERROR: " + message))
            else:
                print("Deletion cancelled")
            
        elif choice == "6":
            success, message = manager.save_contacts()
            print("\n" + ("SUCCESS: " + message if success else "ERROR: " + message))
            print("\nThank you for using Contact Book Application!")
            break
            
        else:
            print("Invalid choice. Please enter a number between 1 and 6")

if __name__ == "__main__":
    main_menu()



CONTACT BOOK APPLICATION
1. Add New Contact
2. View All Contacts (Sorted)
3. Search Contact
4. Update Contact
5. Delete Contact
6. Save & Exit

SUCCESS: Saved 0 contacts

Thank you for using Contact Book Application!
